# Baseline Logistic Regression

전처리된 Train 데이터로 **튜닝하지 않은 기본 LogisticRegression**을 학습하고 Validation 성능을 확인합니다. 최종 XGBoost도 함께 불러와 동일한 Validation 데이터에서 비교합니다.

- 모델 선택이나 튜닝에는 Test 데이터를 사용하지 않습니다.
- 공정한 모델 비교는 두 모델 모두 threshold 0.5 결과를 봅니다.
- XGBoost threshold 0.38은 Validation에서 이미 정한 실제 운영 기준이므로 별도 행으로 표시합니다.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)

def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'data' / 'preprocessed').exists() and (path / 'models').exists():
            return path
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

ROOT = find_project_root()
DATA_DIR = ROOT / 'data' / 'preprocessed'
FINAL_MODEL_PATH = ROOT / 'models' / 'final' / 'model_final.joblib'
RANDOM_STATE = 42
ROOT

In [ ]:
X_train = pd.read_csv(DATA_DIR / 'X_train.csv')
y_train = pd.read_csv(DATA_DIR / 'y_train.csv')['churn']
X_val = pd.read_csv(DATA_DIR / 'X_val.csv')
y_val = pd.read_csv(DATA_DIR / 'y_val.csv')['churn']

print(f'Train: {X_train.shape}, 이탈률={y_train.mean():.3f}')
print(f'Val:   {X_val.shape}, 이탈률={y_val.mean():.3f}')
display(X_train.head())

In [ ]:
# 기본값을 유지하고 수렴을 위해 max_iter만 충분히 크게 설정합니다.
baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
baseline_val_proba = baseline.predict_proba(X_val)[:, 1]
print(baseline)

In [ ]:
def evaluate(name, y_true, probabilities, threshold=0.5):
    pred = (probabilities >= threshold).astype(int)
    return {
        '모델/기준': name,
        'threshold': threshold,
        'Accuracy': accuracy_score(y_true, pred),
        'Recall': recall_score(y_true, pred),
        'Precision': precision_score(y_true, pred, zero_division=0),
        'F1': f1_score(y_true, pred),
        'ROC-AUC': roc_auc_score(y_true, probabilities),
    }

rows = [evaluate('Baseline Logistic', y_val, baseline_val_proba, 0.5)]

if FINAL_MODEL_PATH.exists():
    final_model = joblib.load(FINAL_MODEL_PATH)
    final_val_proba = final_model.predict_proba(X_val)[:, 1]
    rows += [
        evaluate('Final XGBoost (공통 기준)', y_val, final_val_proba, 0.5),
        evaluate('Final XGBoost (운영 기준)', y_val, final_val_proba, 0.38),
    ]
else:
    print(f'최종 모델을 찾지 못해 Baseline만 평가합니다: {FINAL_MODEL_PATH}')

comparison = pd.DataFrame(rows).set_index('모델/기준')
display(comparison.style.format('{:.3f}').highlight_max(
    subset=['Recall', 'Precision', 'F1', 'ROC-AUC'], color='#d9ead3'
))

In [ ]:
baseline_pred = (baseline_val_proba >= 0.5).astype(int)
baseline_cm = pd.DataFrame(
    confusion_matrix(y_val, baseline_pred),
    index=['실제 잔류(0)', '실제 이탈(1)'],
    columns=['예측 잔류(0)', '예측 이탈(1)'],
)
display(baseline_cm)

coef = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': baseline.coef_[0],
}).sort_values('coefficient', key=abs, ascending=False)
display(coef)

## 해석 시 주의

Baseline은 성능의 출발점을 보여주기 위한 모델이며 GridSearch, class_weight 조정, threshold 최적화를 하지 않았습니다. 최종 보고에서는 0.5 기준 비교와 운영 threshold 비교를 혼동하지 않고 따로 설명합니다.